# Gold - fact_vehicle_assembly

This notebook loads `amperedrive_giga.gold.fact_vehicle_assembly`, one row per assembled vehicle (`assembly_id`). Sources are `silver.fact_vehicle_assembly`, filtered to active SCD2 versions (`is_current = TRUE`), joined to `silver.dim_products` for vehicle cost and value.

* Load pattern: Incremental MERGE on the business key, with a watermark on `silver.updated_at` compared with `MAX(gold.gold_load_ts)`.
* DQ-G04 (referential integrity): a NULL `plant_id`, `product_id` or `equipment_id` maps to -1 (unknown member).
* Load after the gold dimension notebooks.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.fact_vehicle_assembly (
  assembly_sk               BIGINT         NOT NULL COMMENT 'PK - xxhash64(assembly_id)',
  assembly_id               INT            NOT NULL COMMENT 'NK / degenerate dimension',
  assembly_date_key         INT            NOT NULL COMMENT 'FK -> dim_date.date_key (COALESCE(end_time, start_time))',
  plant_sk                  BIGINT         NOT NULL COMMENT 'FK -> dim_plant (-1 = unknown)',
  product_sk                BIGINT         NOT NULL COMMENT 'FK -> dim_product (-1 = unknown)',
  equipment_sk              BIGINT         NOT NULL COMMENT 'FK -> dim_equipment (-1 = unknown)',
  vin                       STRING         NOT NULL,
  assembly_line_number      INT            NOT NULL,
  quality_pass_status       STRING         NOT NULL,
  is_passed_flag            INT            NOT NULL,
  assembly_duration_mins    DECIMAL(10,2),
  vehicle_material_cost_usd DECIMAL(38,18) NOT NULL,
  vehicle_output_value_usd  DECIMAL(38,18) NOT NULL,
  gold_load_ts              TIMESTAMP      NOT NULL,
  CONSTRAINT pk_fact_vehicle_assembly PRIMARY KEY (assembly_sk)
)
COMMENT 'Gold vehicle assembly fact (active SCD2 versions from silver.fact_vehicle_assembly)'

In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_watermark TIMESTAMP;
SET VAR v_watermark = (
  SELECT COALESCE(MAX(gold_load_ts), TIMESTAMP'1900-01-01 00:00:00')
  FROM amperedrive_giga.gold.fact_vehicle_assembly
);
SELECT v_watermark AS watermark;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_src_fact_vehicle_assembly AS
SELECT
  xxhash64(CAST(a.assembly_id AS STRING))                                                 AS assembly_sk,
  a.assembly_id,
  CAST(DATE_FORMAT(COALESCE(a.end_time, a.start_time), 'yyyyMMdd') AS INT)                AS assembly_date_key,
  -- DQ-G04: map missing references to the -1 unknown member (xxhash64(NULL) would return 42)
  CASE WHEN a.plant_id     IS NULL THEN -1 ELSE xxhash64(CAST(a.plant_id     AS STRING)) END AS plant_sk,
  CASE WHEN a.product_id   IS NULL THEN -1 ELSE xxhash64(CAST(a.product_id   AS STRING)) END AS product_sk,
  CASE WHEN a.equipment_id IS NULL THEN -1 ELSE xxhash64(CAST(a.equipment_id AS STRING)) END AS equipment_sk,
  UPPER(TRIM(a.vin))                                                                      AS vin,
  a.assembly_line_number,
  UPPER(TRIM(a.quality_pass_status))                                                      AS quality_pass_status,
  CASE WHEN UPPER(TRIM(a.quality_pass_status)) IN ('PASS', 'PASSED', 'SUCCESS') THEN 1 ELSE 0 END AS is_passed_flag,
  CAST(ROUND((UNIX_TIMESTAMP(a.end_time) - UNIX_TIMESTAMP(a.start_time)) / 60.0, 2) AS DECIMAL(10,2)) AS assembly_duration_mins,
  CAST(COALESCE(p.material_cost_usd, 0)       AS DECIMAL(38,18))                          AS vehicle_material_cost_usd,
  CAST(COALESCE(p.factory_output_cost_usd, 0) AS DECIMAL(38,18))                          AS vehicle_output_value_usd,
  CURRENT_TIMESTAMP()                                                                     AS gold_load_ts
FROM amperedrive_giga.silver.fact_vehicle_assembly a
LEFT JOIN amperedrive_giga.silver.dim_products p
  ON a.product_id = p.product_id
WHERE a.is_current = TRUE           -- SCD Type 2 filtering rule: active records only
  AND a.updated_at > v_watermark;

SELECT COUNT(*) AS incoming_rows FROM vw_src_fact_vehicle_assembly;

In [0]:
%sql
-- SCD2 leak guard: MERGE requires a single active version per assembly_id
SELECT
  COUNT(assembly_id)          AS total_assemblies,
  COUNT(DISTINCT assembly_id) AS distinct_assemblies,
  CASE WHEN COUNT(assembly_id) <> COUNT(DISTINCT assembly_id)
       THEN raise_error('Duplicate active assembly_id in silver.fact_vehicle_assembly (SCD2 leak)')
  END AS dup_status
FROM vw_src_fact_vehicle_assembly

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.fact_vehicle_assembly AS t
USING vw_src_fact_vehicle_assembly AS s
ON t.assembly_id = s.assembly_id
WHEN MATCHED AND (
     NOT (t.assembly_date_key         <=> s.assembly_date_key)
  OR NOT (t.plant_sk                  <=> s.plant_sk)
  OR NOT (t.product_sk                <=> s.product_sk)
  OR NOT (t.equipment_sk              <=> s.equipment_sk)
  OR NOT (t.vin                       <=> s.vin)
  OR NOT (t.assembly_line_number      <=> s.assembly_line_number)
  OR NOT (t.quality_pass_status       <=> s.quality_pass_status)
  OR NOT (t.assembly_duration_mins    <=> s.assembly_duration_mins)
  OR NOT (t.vehicle_material_cost_usd <=> s.vehicle_material_cost_usd)
  OR NOT (t.vehicle_output_value_usd  <=> s.vehicle_output_value_usd)
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM amperedrive_giga.gold.fact_vehicle_assembly) AS gold_rows,
  (SELECT COUNT(DISTINCT assembly_id) FROM amperedrive_giga.silver.fact_vehicle_assembly WHERE is_current) AS silver_active_assemblies,
  (SELECT SUM(is_passed_flag) FROM amperedrive_giga.gold.fact_vehicle_assembly) AS passed_vehicles,
  (SELECT COUNT(*) FROM amperedrive_giga.gold.fact_vehicle_assembly WHERE plant_sk = -1 OR product_sk = -1) AS unknown_member_rows,
  assert_true(
    (SELECT COUNT(*) FROM amperedrive_giga.gold.fact_vehicle_assembly WHERE plant_sk IS NULL OR product_sk IS NULL) = 0,
    'DQ-G04 failed: NULL plant_sk/product_sk in gold.fact_vehicle_assembly') AS dq_g04,
  assert_true(
    (SELECT COUNT(assembly_id) - COUNT(DISTINCT assembly_id) FROM amperedrive_giga.gold.fact_vehicle_assembly) = 0,
    'Duplicate assembly_id in gold.fact_vehicle_assembly') AS dq_unique_nk